# 5. Defeitos: vazios no revestimento e uma varredura em linha

Um **vazio** (ar, er = 1) dentro do polímero diminui a permissividade efetiva perto da antena.
A classe `emslab.Defeito` descreve uma caixa de outro material dentro da camada:

```python
emslab.Defeito(largura_x=20, largura_y=20, altura=2, x=0, y=0, z=0, er=1.0, tand=0.0)
```

`z` é a altura da base do defeito a partir do metal. `z = 0` é uma delaminação na interface com o aço.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import emslab

antena = emslab.patch_retangular(f_ghz=3.5)
ESPESSURA, ER, TAND, GAP = 10, 4.0, 0.01, 3


def simular(nome, defeito=None, x_antena=0.0):
    sim = emslab.Simulacao(f_min=2, f_max=5)
    sim.antena(antena, x=x_antena)
    sim.amostra(espessura=ESPESSURA, er=ER, tand=TAND, gap=GAP)
    if defeito is not None:
        sim.defeito(defeito)
    return sim.rodar(nome)

## 5.1 Onde fica o vazio importa?

Três casos, com o vazio de 30 x 30 x 2 mm logo abaixo da antena:
na interface com o metal, no meio da camada e perto da superfície.

In [ ]:
sem_defeito = simular("sem_defeito")
casos = {"vazio na interface (z = 0)": 0, "vazio no meio (z = 4 mm)": 4, "vazio na superfície (z = 8 mm)": 8}
com_defeito = []
for nome, z in casos.items():
    r = simular(nome.split(" (")[0].replace(" ", "_"), emslab.Defeito(largura_x=30, largura_y=30, altura=2, z=z))
    r.nome = nome
    com_defeito.append(r)

sem_defeito.nome = "sem defeito"
emslab.grafico_s11(sem_defeito, *com_defeito)
plt.xlim(3.0, 3.8)
for r in com_defeito:
    print(f"{r.nome}: deslocamento de {(r.f_ressonancia - sem_defeito.f_ressonancia) * 1000:+.1f} MHz")

**Pergunta:** por que o vazio na interface com o metal quase não muda a ressonância?
(Dica: o que acontece com o campo elétrico tangencial na superfície de um condutor?)

## 5.2 Varredura em linha: o primeiro "mapa"

O sistema real move o sensor sobre o tubo e mede em cada ponto. Aqui a antena anda no eixo x,
de −60 mm a +60 mm, sobre um vazio de 30 x 30 x 2 mm perto da superfície, em x = 0.

São 9 simulações. Espere alguns minutos.

In [ ]:
vazio = emslab.Defeito(largura_x=30, largura_y=30, altura=2, z=8)
posicoes = np.arange(-60, 61, 15)
varredura = [simular(f"varredura_x{x:+d}", vazio, x_antena=x) for x in posicoes]

desvio = np.array([(r.f_ressonancia - sem_defeito.f_ressonancia) * 1000 for r in varredura])
fig, ax = plt.subplots(figsize=(8, 4), tight_layout=True)
ax.plot(posicoes, desvio, "o-")
ax.axvspan(-15, 15, color="orange", alpha=0.2, label="posição do vazio")
ax.set_xlabel("Posição da antena x (mm)"); ax.set_ylabel("Δf0 (MHz)")
ax.set_title("Varredura em linha sobre um vazio"); ax.grid(True); ax.legend()
fig.savefig("05_varredura_vazio.png", dpi=150)

## 5.3 Próximos passos

- Troque o vazio por **umidade** (`er=78, tand=0.1`) e compare o sinal com o do vazio.
- Repita a varredura em y e monte um mapa 2D com `plt.pcolormesh`.
- Compare a resolução espacial (largura do pico) com o tamanho do patch. Como deixar o sensor mais "fino"?